# Respiratory sound GNN — training on IBM LinuxONE

This notebook trains the graph neural network served by the web app, end to end, on the machine it runs on.
The model was first prototyped in Google Colab (`model_trained.ipynb`); this is the LinuxONE version of that
training run, with three deliberate changes:

| | Colab prototype | This notebook |
|---|---|---|
| Train/test split | random, by recording | **by patient** — no patient appears in both training and test data |
| Best-epoch selection | chosen on the test set | chosen on a separate **validation** set; the test set is used once, at the end |
| Feature extraction | recomputed every epoch | computed **once** and cached |

Because of the first two changes, expect a lower test accuracy than the prototype's 95.47%: that number was
optimistic, because recordings of the same patient were in both sets and the test set also picked the best epoch.
The number reported here is the one to quote.

Run all cells top to bottom. The final cells save the model in the exact format the app loads and check it with
the app's own inference code.

## 1. Environment

In [ ]:
import os, platform, re, sys, time, json, copy, pickle, shutil, datetime as dt
from pathlib import Path

import numpy as np
import librosa
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

def _cpuinfo():
    try:
        return Path("/proc/cpuinfo").read_text()
    except OSError:
        return ""

def _mem_gb():
    try:
        kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
        return kb / 1024 / 1024
    except (OSError, AttributeError):
        return float("nan")

cpuinfo = _cpuinfo()
machine_type = re.search(r"machine\s*=\s*(\w+)", cpuinfo)      # IBM Z machine type, e.g. 3931 = z16
features_line = re.search(r"^features\s*:\s*(.*)$", cpuinfo, re.M)
has_nnpa = bool(features_line and "nnpa" in features_line.group(1).split())

RUN_STARTED = dt.datetime.now()
ENV = {
    "host": platform.node(),
    "architecture": platform.machine(),
    "byte_order": sys.byteorder,
    "ibm_z_machine_type": machine_type.group(1) if machine_type else None,
    "nnpa_ai_accelerator": has_nnpa,
    "cpu_cores": os.cpu_count(),
    "memory_gb": round(_mem_gb(), 1),
    "os": platform.platform(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "librosa": librosa.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "started": RUN_STARTED.isoformat(timespec="seconds"),
}
for k, v in ENV.items():
    print(f"{k:22s} {v}")
if ENV["architecture"] != "s390x":
    print("\nNote: not running on IBM Z / LinuxONE (s390x).")

## 2. Configuration

In [ ]:
REPO = Path.cwd()
DATASET = REPO / "Asthma Detection Dataset Version 2"
APP_ML_DIR = REPO / "respiratory-sound-classifier" / "backend" / "python_service"
CACHE_DIR = REPO / "cache"
assert DATASET.is_dir(), f"Dataset not found at {DATASET} - start Jupyter from the repository root"

# SMOKE_TEST=1 runs a tiny version (few patients, 3 epochs) to check the pipeline quickly
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"

SEED = 42
TARGET_SR = 16000        # Hz - must match the deployed service (predict.py)
DURATION = 5.0           # seconds - centre crop / zero pad
K_NEIGHBORS = 5          # frames linked per node in the kNN graph

EPOCHS = 3 if SMOKE_TEST else 100
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 20            # early stopping on validation accuracy
MODEL_CONFIG = {"input_dim": 76, "hidden_dim": 128, "num_classes": 5, "dropout": 0.5}

np.random.seed(SEED)
torch.manual_seed(SEED)
# These graphs are small: beyond a few threads, PyTorch spends more time coordinating than computing
TORCH_THREADS = int(os.environ.get("TORCH_THREADS", min(4, os.cpu_count())))
torch.set_num_threads(TORCH_THREADS)
print(f"Smoke test: {SMOKE_TEST} | epochs: {EPOCHS} | torch threads: {torch.get_num_threads()}")

## 3. Dataset and patients

File names start with a patient number (`P10AsthmaIU_50.wav` → patient 10). Patient numbers restart in each
class folder, so a patient is identified by *folder + number* (`asthma/P10`). These patient IDs are what keeps
the split honest.

In [ ]:
CLASS_FOLDERS = {"Bronchial": "Bronchial", "asthma": "Asthma", "copd": "COPD", "healthy": "Healthy", "pneumonia": "Pneumonia"}

records = []
for folder, label in CLASS_FOLDERS.items():
    for f in sorted((DATASET / folder).glob("*.wav")):
        m = re.match(r"P(\d+)", f.name)
        patient = f"{folder}/P{int(m.group(1))}" if m else f"{folder}/{f.stem}"   # unnumbered files: own group
        records.append({"path": f, "label": label, "patient": patient})

if SMOKE_TEST:  # keep the first 6 patients of each class
    keep = set()
    for label in CLASS_FOLDERS.values():
        pats = sorted({r["patient"] for r in records if r["label"] == label})[:6]
        keep.update(pats)
    records = [r for r in records if r["patient"] in keep]

paths = [r["path"] for r in records]
labels = np.array([r["label"] for r in records])
patients = np.array([r["patient"] for r in records])

print(f"{'class':10s} {'recordings':>10s} {'patients':>9s}")
for label in sorted(set(labels)):
    sel = labels == label
    print(f"{label:10s} {sel.sum():10d} {len(set(patients[sel])):9d}")
print(f"{'total':10s} {len(records):10d} {len(set(patients)):9d}")

## 4. Acoustic features (computed once, cached)

The exact preprocessing and feature code used by the deployed inference service: resample to 16 kHz, centre
5 s, peak-normalise, then 76 features per frame (MFCC + Δ + Δ², spectral centroid/roll-off/contrast/bandwidth,
zero-crossing rate, chroma, 13 mel bands, RMS) → a 76 × 157 matrix per recording.

In [ ]:
def load_audio(path):
    audio, sr = librosa.load(path, sr=TARGET_SR)
    n = int(TARGET_SR * DURATION)
    if len(audio) > n:
        start = (len(audio) - n) // 2
        audio = audio[start:start + n]
    elif len(audio) < n:
        audio = np.pad(audio, (0, n - len(audio)), mode="constant")
    peak = np.max(np.abs(audio))
    return audio / peak if peak > 0 else audio

def extract_features(audio, sr=TARGET_SR):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=13), ref=np.max)
    return np.vstack([
        mfcc,
        librosa.feature.delta(mfcc),
        librosa.feature.delta(mfcc, order=2),
        librosa.feature.spectral_centroid(y=audio, sr=sr),
        librosa.feature.spectral_rolloff(y=audio, sr=sr),
        librosa.feature.spectral_contrast(y=audio, sr=sr),
        librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        librosa.feature.zero_crossing_rate(audio),
        librosa.feature.chroma_stft(y=audio, sr=sr),
        mel,
        librosa.feature.rms(y=audio),
    ]).astype(np.float32)

def features_for(path):
    return extract_features(load_audio(path))

CACHE_DIR.mkdir(exist_ok=True)
cache_file = CACHE_DIR / f"features_{'smoke_' if SMOKE_TEST else ''}{TARGET_SR}hz_{DURATION:g}s.npz"
rel_paths = np.array([str(p.relative_to(REPO)) for p in paths])

if cache_file.exists() and np.array_equal(np.load(cache_file)["paths"], rel_paths):
    X = np.load(cache_file)["features"]
    FEATURE_SECONDS = None
    print(f"Loaded cached features {X.shape} from {cache_file.name}")
else:
    from joblib import Parallel, delayed
    t0 = time.perf_counter()
    # Run one file here first: on x86_64 this compiles and caches librosa's numba kernels once, so the
    # worker processes don't all compile and write that cache at the same time (which can crash them).
    features_for(paths[0])
    try:
        X = np.stack(Parallel(n_jobs=os.cpu_count())(delayed(features_for)(p) for p in paths))
    except Exception as e:  # never lose a long run to a worker problem: fall back to one process
        print(f"Parallel extraction failed ({type(e).__name__}); continuing in a single process")
        X = np.stack([features_for(p) for p in paths])
    FEATURE_SECONDS = time.perf_counter() - t0
    np.savez_compressed(cache_file, features=X, paths=rel_paths)
    print(f"Extracted features {X.shape} for {len(paths)} recordings in {FEATURE_SECONDS:.1f} s "
          f"({len(paths) / FEATURE_SECONDS:.1f} recordings/s on {os.cpu_count()} cores)")

## 5. Recordings as graphs

Each of the 157 frames becomes a node with its 76 features; every node is connected to its 5 most similar frames
(Euclidean distance), in both directions. Same construction as the inference service.

In [ ]:
def knn_edges(nodes, k=K_NEIGHBORS):
    n = nodes.shape[0]
    if n <= 1:
        return np.array([[0], [0]])
    dist = np.linalg.norm(nodes[:, None] - nodes[None, :], axis=2)
    edges = set()
    for i in range(n):
        for j in np.argsort(dist[i])[1:min(k, n - 1) + 1]:
            edges.add((i, int(j)))
            edges.add((int(j), i))
    return np.array(sorted(edges)).T

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder().fit(sorted(set(labels)))
y = label_encoder.transform(labels)

t0 = time.perf_counter()
graphs = []
for feats, target in zip(X, y):
    nodes = feats.T                                   # (157 frames, 76 features)
    graphs.append(Data(x=torch.from_numpy(nodes.copy()),
                       edge_index=torch.from_numpy(knn_edges(nodes)).long(),
                       y=torch.tensor([target])))
print(f"Built {len(graphs)} graphs in {time.perf_counter() - t0:.1f} s — "
      f"{graphs[0].num_nodes} nodes, ~{np.mean([g.num_edges for g in graphs]):.0f} directed edges each")
print("Classes:", ", ".join(map(str, label_encoder.classes_)))

## 6. Patient-level split

Patients (not recordings) are divided into train / validation / test, stratified by class:
roughly 64% / 16% / 20% of patients. The test set is not touched until section 9.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

idx = np.arange(len(graphs))
n_splits = 3 if SMOKE_TEST else 5
outer = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(outer.split(idx, y, patients))
inner = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
tr, va = next(inner.split(trainval_idx, y[trainval_idx], patients[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

sets = {"train": train_idx, "validation": val_idx, "test": test_idx}
for a in sets:
    for b in sets:
        if a < b:
            assert not set(patients[sets[a]]) & set(patients[sets[b]]), f"patient leak between {a} and {b}"
print("No patient appears in more than one split.\n")

print(f"{'split':11s} {'recordings':>10s} {'patients':>9s}   " + "  ".join(f"{c:>9s}" for c in label_encoder.classes_))
for name, s in sets.items():
    per_class = [int((y[s] == k).sum()) for k in range(len(label_encoder.classes_))]
    print(f"{name:11s} {len(s):10d} {len(set(patients[s])):9d}   " + "  ".join(f"{n:9d}" for n in per_class))

## 7. Model

In [ ]:
class MultiClassRespiratoryGNN(nn.Module):
    """Identical to the class in the inference service (predict.py)."""

    def __init__(self, input_dim=76, hidden_dim=128, num_classes=5, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.conv4 = GCNConv(hidden_dim, hidden_dim // 2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.bn4 = nn.BatchNorm1d(hidden_dim // 2)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim // 2, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(dropout * 0.7),
            nn.Linear(64, 32), nn.ReLU(), nn.BatchNorm1d(32), nn.Dropout(dropout * 0.5),
            nn.Linear(32, num_classes),
        )

    def forward(self, x, edge_index, batch):
        x1 = self.dropout(F.relu(self.bn1(self.conv1(x, edge_index))))
        x2 = self.dropout(F.relu(self.bn2(self.conv2(x1, edge_index)))) + x1
        x3 = self.dropout(F.relu(self.bn3(self.conv3(x2, edge_index)))) + x2
        x4 = F.relu(self.bn4(self.conv4(x3, edge_index)))
        return F.log_softmax(self.classifier(global_mean_pool(x4, batch)), dim=1)

model = MultiClassRespiratoryGNN(**MODEL_CONFIG)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Training (on this machine's CPU)

In [ ]:
def subset(indices):
    return [graphs[i] for i in indices]

torch.manual_seed(SEED)
train_loader = DataLoader(subset(train_idx), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(subset(val_idx), batch_size=64)
test_loader = DataLoader(subset(test_idx), batch_size=64)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=10)
criterion = nn.NLLLoss()

def evaluate(loader):
    model.eval()
    preds, targets, loss_sum = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            out = model(batch.x, batch.edge_index, batch.batch)
            loss_sum += criterion(out, batch.y).item() * batch.num_graphs
            preds.append(out.argmax(1))
            targets.append(batch.y)
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return loss_sum / len(targets), (preds == targets).mean(), preds, targets

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_seconds": []}
best = {"val_acc": -1.0, "epoch": -1, "state": None}
wait = 0
train_started = time.perf_counter()

for epoch in range(EPOCHS):
    t0 = time.perf_counter()
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item() * batch.num_graphs
        correct += (out.argmax(1) == batch.y).sum().item()
        seen += batch.num_graphs
    val_loss, val_acc, _, _ = evaluate(val_loader)
    scheduler.step(val_acc)

    history["train_loss"].append(loss_sum / seen)
    history["train_acc"].append(correct / seen)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(float(val_acc))
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["epoch_seconds"].append(time.perf_counter() - t0)

    if val_acc > best["val_acc"]:
        best = {"val_acc": float(val_acc), "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
        wait = 0
    else:
        wait += 1

    if epoch % 5 == 0 or wait == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train loss {history['train_loss'][-1]:.4f}  train acc {history['train_acc'][-1]:.3f}  "
              f"val acc {val_acc:.3f}  lr {history['lr'][-1]:.6f}  {history['epoch_seconds'][-1]:.1f}s"
              + ("  ← best" if wait == 0 else ""))
    if wait >= PATIENCE:
        print(f"Early stopping: no validation improvement for {PATIENCE} epochs")
        break

TRAIN_SECONDS = time.perf_counter() - train_started
model.load_state_dict(best["state"])
print(f"\nTrained {len(history['val_acc'])} epochs in {TRAIN_SECONDS / 60:.1f} min on {ENV['architecture']} "
      f"({np.mean(history['epoch_seconds']):.1f} s/epoch). Best validation accuracy {best['val_acc']:.3f} at epoch {best['epoch']}.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="validation")
ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()
ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"], label="validation")
ax2.axvline(best["epoch"], color="grey", ls="--", lw=1, label="selected epoch")
ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
fig.suptitle(f"Training on {ENV['architecture']} ({ENV['cpu_cores']} cores)")
plt.tight_layout()
plt.show()

## 9. Test on unseen patients

The held-out patients are evaluated once, with the epoch chosen on validation data. For a screening tool,
**recall** per class matters most: it is the share of patients with that condition the model catches.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

_, _, test_pred, test_true = evaluate(test_loader)
classes = list(label_encoder.classes_)
labels_present = list(range(len(classes)))
METRICS = {
    "test_accuracy": float(accuracy_score(test_true, test_pred)),
    "test_balanced_accuracy": float(balanced_accuracy_score(test_true, test_pred)),
    "test_macro_f1": float(f1_score(test_true, test_pred, average="macro", labels=labels_present, zero_division=0)),
    "test_recordings": int(len(test_true)),
    "test_patients": int(len(set(patients[test_idx]))),
}
for k, v in METRICS.items():
    print(f"{k:24s} {v:.3f}" if isinstance(v, float) else f"{k:24s} {v}")
print()
print(classification_report(test_true, test_pred, labels=labels_present, target_names=classes, digits=3, zero_division=0))

cm = confusion_matrix(test_true, test_pred, labels=labels_present)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Test set (unseen patients)")
plt.tight_layout()
plt.show()

## 10. Save the model

Saved in the format the inference service loads (`model.pth`, `metadata.json`, `label_encoder.pkl`), together with
the training history and a record of the environment it was trained in.

In [ ]:
stamp = RUN_STARTED.strftime("%Y%m%d_%H%M%S")
OUT = REPO / "saved_models" / f"respiratory_model_{ENV['architecture']}_{stamp}{'_smoke' if SMOKE_TEST else ''}"
OUT.mkdir(parents=True, exist_ok=True)

torch.save({"model_state_dict": model.state_dict(), "epoch": best["epoch"], "val_accuracy": best["val_acc"]}, OUT / "model.pth")
with open(OUT / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

metadata = {
    "model_config": MODEL_CONFIG,
    "training": {
        "environment": ENV,
        "finished": dt.datetime.now().isoformat(timespec="seconds"),
        "feature_extraction_seconds": FEATURE_SECONDS,
        "training_seconds": round(TRAIN_SECONDS, 1),
        "epochs_run": len(history["val_acc"]),
        "selected_epoch": best["epoch"],
        "validation_accuracy": best["val_acc"],
        "split": "patient-level StratifiedGroupKFold (train/validation/test)",
        "recordings": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
        "torch_threads": TORCH_THREADS,
        "hyperparameters": {"batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY, "patience": PATIENCE,
                            "k_neighbors": K_NEIGHBORS, "sample_rate": TARGET_SR, "duration_s": DURATION, "seed": SEED},
        **METRICS,
    },
}
(OUT / "metadata.json").write_text(json.dumps(metadata, indent=2))
(OUT / "training_history.json").write_text(json.dumps(history, indent=2))
for name in sorted(os.listdir(OUT)):
    print(f"{name:24s} {(OUT / name).stat().st_size:>9,d} bytes")
print("\nSaved to", OUT.relative_to(REPO))

## 11. Check the model with the app's own inference code

Loads the saved files through the deployed service's `predict.py` — the same code path the web app uses — and
classifies the five example recordings bundled with the frontend. (Those examples come from the dataset, so
some may have been in the training split; this cell checks compatibility, not accuracy.)

In [ ]:
sys.path.insert(0, str(APP_ML_DIR))
os.environ["MODEL_DIR"] = str(OUT)
import predict as service

assert service.load_model(), "the inference service could not load the new model"
samples_dir = REPO / "respiratory-sound-classifier" / "frontend" / "public" / "samples"
for wav in sorted(samples_dir.glob("*.wav")):
    r = service.classify_file(str(wav))
    print(f"{wav.stem:10s} -> {r['predicted_class']:10s} {r['confidence']:.3f}   ({r['inference_ms']:.0f} ms on {r['arch']})")

## 12. Deploy this model (optional)

Set `DEPLOY = True` and run the cell to make this the model the web app serves. Then rebuild on the server:

```bash
cd ~/IBM-AI-Powered-Respiratory-Disease-Detection/respiratory-sound-classifier && bash deploy/deploy.sh
```

In [ ]:
DEPLOY = False

target = APP_ML_DIR / "model"
if DEPLOY and not SMOKE_TEST:
    for name in ("model.pth", "metadata.json", "label_encoder.pkl"):
        shutil.copy2(OUT / name, target / name)
    print(f"Copied {OUT.name} into {target.relative_to(REPO)} — now run deploy/deploy.sh")
else:
    print("Not deployed (set DEPLOY = True to copy this model into the app).")

## Summary

In [ ]:
print(f"Trained on      : {ENV['host']} — {ENV['architecture']}"
      + (f" (IBM Z machine type {ENV['ibm_z_machine_type']})" if ENV["ibm_z_machine_type"] else "")
      + f", {ENV['cpu_cores']} cores, {ENV['memory_gb']} GB")
print(f"AI accelerator  : {'NNPA present' if ENV['nnpa_ai_accelerator'] else 'NNPA not reported by this CPU'} (training used the CPU)")
print(f"Data            : {len(graphs)} recordings from {len(set(patients))} patients, patient-level split")
print(f"Features        : " + (f"{FEATURE_SECONDS:.0f} s to extract" if FEATURE_SECONDS else "loaded from cache"))
print(f"Training        : {len(history['val_acc'])} epochs in {TRAIN_SECONDS / 60:.1f} min")
print(f"Test (unseen patients): accuracy {METRICS['test_accuracy']:.3f}, balanced accuracy "
      f"{METRICS['test_balanced_accuracy']:.3f}, macro F1 {METRICS['test_macro_f1']:.3f}")
print(f"Model saved     : {OUT.relative_to(REPO)}")